# 第2节：视频的色彩空间 - 动手实验

本notebook包含第2节课的所有实验代码，请按顺序执行每个单元。

## 1. 导入必要的库

In [ ]:
import av
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

print("✅ 所有库导入成功！")
print(f"PyAV版本: {av.__version__}")
print(f"OpenCV版本: {cv2.__version__}")

## 2. 打开视频文件并读取第一帧

In [ ]:
# 视频文件路径
video_path = '../assets/video/test_360p_5s.mp4'

# 检查文件是否存在
if not os.path.exists(video_path):
    print("⚠️  测试视频不存在，请先下载：")
    print("mkdir -p ../assets/video")
    print("wget https://sample-videos.com/video123/mp4/360/big_buck_bunny_360p_1mb.mp4 -O ../assets/video/test_360p_5s.mp4")
else:
    # 打开视频文件
    container = av.open(video_path)
    
    # 获取视频流
    video_stream = container.streams.video[0]
    print(f"✅ 视频打开成功！")
    print(f"📹 视频信息：")
    print(f"  分辨率: {video_stream.width}x{video_stream.height}")
    print(f"  帧率: {float(video_stream.average_rate):.2f} fps")
    print(f"  像素格式: {video_stream.pix_fmt}")
    print(f"  总帧数: {video_stream.frames}")
    
    # 读取第一帧
    frame = next(container.decode(video=0))
    print(f"\n🖼️  帧信息：")
    print(f"  格式: {frame.format.name}")
    print(f"  平面数: {len(frame.planes)}")
    print(f"  Y平面步长: {frame.planes[0].line_size}")
    print(f"  U平面步长: {frame.planes[1].line_size}")
    print(f"  V平面步长: {frame.planes[2].line_size}")
    
    # 关闭容器
    container.close()

## 3. 分离Y/U/V分量并可视化

In [ ]:
def read_plane(plane, width, height):
    """读取平面数据为NumPy数组"""
    return np.frombuffer(plane, np.uint8).reshape(height, width)

# 获取YUV分量
y_width, y_height = frame.width, frame.height
uv_width, uv_height = y_width // 2, y_height // 2

y = read_plane(frame.planes[0], y_width, y_height)
u = read_plane(frame.planes[1], uv_width, uv_height)
v = read_plane(frame.planes[2], uv_width, uv_height)

print(f"✅ 分量提取成功！")
print(f"  Y分量尺寸: {y.shape}")
print(f"  U分量尺寸: {u.shape}")
print(f"  V分量尺寸: {v.shape}")

# 可视化三个分量
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Y分量（灰度图）
axes[0].imshow(y, cmap='gray')
axes[0].set_title('Y Component (Luma - Brightness)', fontsize=14)
axes[0].axis('off')

# U分量（蓝色色度，使用冷色调调色板）
axes[1].imshow(u, cmap='Blues')
axes[1].set_title('U Component (Cb - Blue Chroma)', fontsize=14)
axes[1].axis('off')

# V分量（红色色度，使用暖色调调色板）
axes[2].imshow(v, cmap='Reds')
axes[2].set_title('V Component (Cr - Red Chroma)', fontsize=14)
axes[2].axis('off')

plt.tight_layout()
plt.show()

# 对比存储空间
rgb_size = y_width * y_height * 3
yuv_size = y.size + u.size + v.size
print(f"\n💾 存储空间对比：")
print(f"  RGB格式: {rgb_size:,} 字节 ({rgb_size/1024/1024:.2f} MB)")
print(f"  YUV420P格式: {yuv_size:,} 字节 ({yuv_size/1024/1024:.2f} MB)")
print(f"  节省空间: {100 - (yuv_size / rgb_size) * 100:.1f}%")

## 4. 修改U/V分量观察色彩变化

In [ ]:
def yuv420p_to_rgb(y, u, v):
    """简化的YUV420P转RGB，使用BT.601标准"""
    # 上采样U/V到Y的尺寸
    u_upsampled = cv2.resize(u, (y.shape[1], y.shape[0]), interpolation=cv2.INTER_NEAREST)
    v_upsampled = cv2.resize(v, (y.shape[1], y.shape[0]), interpolation=cv2.INTER_NEAREST)
    
    # 转换公式（BT.601）
    y = y.astype(np.float32)
    u = u_upsampled.astype(np.float32) - 128
    v = v_upsampled.astype(np.float32) - 128
    
    r = y + 1.13983 * v
    g = y - 0.39465 * u - 0.58060 * v
    b = y + 2.03211 * u
    
    # 裁剪到0-255范围
    rgb = np.stack([r, g, b], axis=-1)
    rgb = np.clip(rgb, 0, 255).astype(np.uint8)
    return rgb

# 原始RGB图像
rgb_original = yuv420p_to_rgb(y, u, v)

# 版本1：U分量全部置128（无色度）
u_128 = np.full_like(u, 128)
rgb_u128 = yuv420p_to_rgb(y, u_128, v)

# 版本2：V分量全部置128（无色度）
v_128 = np.full_like(v, 128)
rgb_v128 = yuv420p_to_rgb(y, u, v_128)

# 版本3：U/V分量都置128（完全灰度图）
u_all_128 = np.full_like(u, 128)
v_all_128 = np.full_like(v, 128)
rgb_gray = yuv420p_to_rgb(y, u_all_128, v_all_128)

print(f"✅ 分量修改完成，正在可视化...")

# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(rgb_original)
axes[0, 0].set_title('1. Original Image', fontsize=14)
axes[0, 0].axis('off')

axes[0, 1].imshow(rgb_u128)
axes[0, 1].set_title('2. U Component = 128 (Missing Blue)', fontsize=14)
axes[0, 1].axis('off')

axes[1, 0].imshow(rgb_v128)
axes[1, 0].set_title('3. V Component = 128 (Missing Red)', fontsize=14)
axes[1, 0].axis('off')

axes[1, 1].imshow(rgb_gray)
axes[1, 1].set_title('4. U/V = 128 (Grayscale Image)', fontsize=14)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## 5. 调整Y分量亮度

In [ ]:
def adjust_brightness(y, offset):
    """调整Y分量亮度"""
    y_new = y.astype(np.int16) + offset
    return np.clip(y_new, 0, 255).astype(np.uint8)

# 生成不同亮度版本
y_dark = adjust_brightness(y, -80)   # 变暗
y_bright = adjust_brightness(y, 80)  # 变亮

# 转换为RGB
rgb_dark = yuv420p_to_rgb(y_dark, u, v)
rgb_bright = yuv420p_to_rgb(y_bright, u, v)

print(f"✅ 亮度调整完成，正在可视化...")

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb_dark)
axes[0].set_title('Brightness -80', fontsize=14)
axes[0].axis('off')

axes[1].imshow(rgb_original)
axes[1].set_title('Original Brightness', fontsize=14)
axes[1].axis('off')

axes[2].imshow(rgb_bright)
axes[2].set_title('Brightness +80', fontsize=14)
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 6. 保存修改后的图像

In [ ]:
# 确保输出目录存在
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

# 保存不同版本的图片（注意OpenCV使用BGR顺序）
cv2.imwrite(os.path.join(output_dir, 'yuv_original.jpg'), cv2.cvtColor(rgb_original, cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(output_dir, 'yuv_u128.jpg'), cv2.cvtColor(rgb_u128, cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(output_dir, 'yuv_v128.jpg'), cv2.cvtColor(rgb_v128, cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(output_dir, 'yuv_gray.jpg'), cv2.cvtColor(rgb_gray, cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(output_dir, 'yuv_dark.jpg'), cv2.cvtColor(rgb_dark, cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(output_dir, 'yuv_bright.jpg'), cv2.cvtColor(rgb_bright, cv2.COLOR_RGB2BGR))

print(f"✅ 所有图片已保存到 {output_dir} 目录：")
print(f"  - yuv_original.jpg  (原始图像)")
print(f"  - yuv_u128.jpg      (U分量置128)")
print(f"  - yuv_v128.jpg      (V分量置128)")
print(f"  - yuv_gray.jpg      (灰度图像)")
print(f"  - yuv_dark.jpg      (亮度-80)")
print(f"  - yuv_bright.jpg    (亮度+80)")

## 🎯 课后挑战：尝试以下任务

### 挑战1：修改指定区域的色度
将画面中心200x200区域的U/V分量都置为128，让中心区域变成灰度，周围保持彩色。

```python
# 实现代码：
y_challenge = y.copy()
u_challenge = u.copy()
v_challenge = v.copy()

# 计算中心区域（注意U/V尺寸是Y的1/2）
center_y, center_x = y_height // 2, y_width // 2
half_size = 100

# 修改Y对应的U/V区域
u_center_y, u_center_x = center_y // 2, center_x // 2
u_half_size = half_size // 2

u_challenge[u_center_y - u_half_size : u_center_y + u_half_size, 
            u_center_x - u_half_size : u_center_x + u_half_size] = 128
v_challenge[u_center_y - u_half_size : u_center_y + u_half_size, 
            u_center_x - u_half_size : u_center_x + u_half_size] = 128

# 转换为RGB并显示
rgb_challenge = yuv420p_to_rgb(y_challenge, u_challenge, v_challenge)
plt.figure(figsize=(12, 7))
plt.imshow(rgb_challenge)
plt.title('Center Region Grayscale', fontsize=14)
plt.axis('off')
plt.show()
```

### 挑战2：手动实现U/V上采样
不使用OpenCV的resize函数，手动实现U/V分量的上采样（最近邻插值）。

```python
def manual_upsample(plane, scale=2):
    """手动实现上采样"""
    h, w = plane.shape
    upsampled = np.zeros((h * scale, w * scale), dtype=np.uint8)
    
    for i in range(h):
        for j in range(w):
            upsampled[i*scale : (i+1)*scale, j*scale : (j+1)*scale] = plane[i, j]
    
    return upsampled

# 测试上采样
u_upsampled_manual = manual_upsample(u)
print(f"原始U尺寸: {u.shape}, 上采样后: {u_upsampled_manual.shape}")
```

### 挑战3：思考题
1. 对于1920x1080的图像，YUV422P需要多少存储空间？比YUV420P多多少？
2. 查找NV12格式的存储结构，思考它和YUV420P的区别和优缺点。

## 📚 总结

恭喜完成第2节课的学习！你已经掌握了：

- ✅ YUV色彩模型的原理和优势
- ✅ YUV420P的采样格式和平面存储结构
- ✅ Y/U/V三个分量的作用：Y控制亮度，U/V控制色度
- ✅ YUV420P比RGB节省50%存储空间
- ✅ 用PyAV读取视频帧和提取YUV分量
- ✅ 修改YUV分量并观察色彩变化
- ✅ YUV到RGB的转换原理

**下节课预告：** 第3节 - 数字音频基础，我们将学习音频的数字化原理和处理方法！🚀